In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import time

In [3]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from moogp.model import MOOGP, init_phi

In [4]:
def generate_synthetic_data(n, d, p, seed=42):
    """Generates a dataset with correlated outputs so SVD works well."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1.0, 1.0, size=(n, d))
    
    # FIX: Stack the two features horizontally to make an (n x 2) matrix
    latent_f = np.column_stack([
        np.sin(3 * X[:, 0]), 
        np.cos(5 * X[:, -1])
    ])
    
    # Mix them into p outputs -> (n x 2) @ (2 x p) = (n x p)
    mixing = rng.normal(size=(2, p))
    Y_clean = latent_f @ mixing
    
    # Add noise
    Y = Y_clean + rng.normal(0, 0.1, size=(n, p))
    return {"X_scaled": X, "Y": Y}

In [5]:
def run_training_benchmark(n, d, p, q, maxiter=150):
    print(f"\n{'='*60}")
    print(f"BENCHMARK: n = {n} | p = {p} | d = {d} | q = {q}")
    print(f"{'='*60}")

    # 1. Generate Data
    data = generate_synthetic_data(n, d, p)
    X = data["X_scaled"]
    Y = data["Y"]

    # 2. Setup Initial Parameters
    y_var = Y.var(axis=0, ddof=1)
    sigma_eps2_init = np.clip(1e-2 * y_var, 1e-6, None)

    # Build the exact same Psi_fair for both models
    Phi, _ = init_phi(Y, q, n)
    Psi_fair = np.diag(np.sqrt(sigma_eps2_init)) @ Phi

    # Setup Theta0 and Bounds
    theta0 = []
    bounds = []
    for _ in range(q):
        theta0.append(np.log(1.0)) # log_s2
        theta0.extend([np.log(0.5)] * d) # log_ell
        bounds.append((np.log(1e-3), np.log(1e3)))
        bounds.extend([(np.log(0.05), np.log(5.0))] * d)

    # Add sigma_eps2 to theta0
    theta0 = np.concatenate([theta0, np.log(sigma_eps2_init)])
    lb = np.maximum(1e-12, 1e-6 * y_var)
    ub = np.maximum(lb * 10.0, 0.5 * y_var)
    bounds.extend([(float(np.log(lbi)), float(np.log(ubi))) for lbi, ubi in zip(lb, ub)])

    # ---------------------------------------------------------
    # TRAIN: SLOW METHOD
    # ---------------------------------------------------------
    print(f"Training SLOW Method (Matrix Size: {n*p} x {n*p})...")
    model_slow = MOOGP(
        terms=[None, 1], q=q, Psi=Psi_fair, learn_Psi=False,
        learn_sigma_eps=True, use_diagonalized_interaction=False, jitter=1e-6
    )
    
    start = time.perf_counter()
    model_slow.fit(data, theta0, bounds=bounds, optimizer_opts={"maxiter": maxiter})
    time_slow = time.perf_counter() - start
    
    iters_slow = model_slow.opt_result.nit
    time_per_iter_slow = time_slow / max(1, iters_slow)

    print(f"  Total Time:   {time_slow:.2f} seconds")
    print(f"  Iterations:   {iters_slow}")
    print(f"  Time/Iter:    {time_per_iter_slow:.4f} s/iter")

    # ---------------------------------------------------------
    # TRAIN: FAST METHOD
    # ---------------------------------------------------------
    print(f"\nTraining FAST Method (Max Matrix Size: {n} x {n})...")
    model_fast = MOOGP(
        terms=[None, 1], q=q, Psi=Psi_fair, learn_Psi=False,
        learn_sigma_eps=True, use_diagonalized_interaction=True, jitter=1e-6
    )
    
    start = time.perf_counter()
    model_fast.fit(data, theta0, bounds=bounds, optimizer_opts={"maxiter": maxiter})
    time_fast = time.perf_counter() - start
    
    iters_fast = model_fast.opt_result.nit
    time_per_iter_fast = time_fast / max(1, iters_fast)

    print(f"  Total Time:   {time_fast:.2f} seconds")
    print(f"  Iterations:   {iters_fast}")
    print(f"  Time/Iter:    {time_per_iter_fast:.4f} s/iter")

    # ---------------------------------------------------------
    # COMPARISON
    # ---------------------------------------------------------
    print("\n--- PERFORMANCE WINNER ---")
    if time_per_iter_slow > time_per_iter_fast:
        speedup = time_per_iter_slow / time_per_iter_fast
        print(f"Fast Method is {speedup:.1f}x faster PER ITERATION.")
    else:
        speedup = time_per_iter_fast / time_per_iter_slow
        print(f"Slow Method is {speedup:.1f}x faster PER ITERATION.")
        
    if time_slow > time_fast:
        print(f"Fast Method finished overall training {time_slow / time_fast:.1f}x faster.")
    else:
        print(f"Slow Method finished overall training {time_fast / time_slow:.1f}x faster.")

In [12]:
run_training_benchmark(n=100, d=4, p=4, q=3)


BENCHMARK: n = 100 | p = 4 | d = 4 | q = 3
Training SLOW Method (Matrix Size: 400 x 400)...
  Total Time:   1.41 seconds
  Iterations:   26
  Time/Iter:    0.0543 s/iter

Training FAST Method (Max Matrix Size: 100 x 100)...
  Total Time:   2.47 seconds
  Iterations:   62
  Time/Iter:    0.0399 s/iter

--- PERFORMANCE WINNER ---
Fast Method is 1.4x faster PER ITERATION.
Slow Method finished overall training 1.8x faster.


In [38]:
# Scenario 2: Medium Data (The algorithmic crossover point)
run_training_benchmark(n=250, d=3, p=8, q=3)


BENCHMARK: n = 250 | p = 8 | d = 3 | q = 3
Training SLOW Method (Matrix Size: 2000 x 2000)...
  Total Time:   146.82 seconds
  Iterations:   58
  Time/Iter:    2.5314 s/iter

Training FAST Method (Max Matrix Size: 250 x 250)...
  Total Time:   134.54 seconds
  Iterations:   150
  Time/Iter:    0.8969 s/iter

--- PERFORMANCE WINNER ---
Fast Method is 2.8x faster PER ITERATION.
Fast Method finished overall training 1.1x faster.


In [10]:
# Scenario 3: Large Data (The Fast Method completely dominates)
run_training_benchmark(n=500, d=4, p=3, q=3)


BENCHMARK: n = 500 | p = 3 | d = 4 | q = 3
Training SLOW Method (Matrix Size: 1500 x 1500)...
  Total Time:   115.09 seconds
  Iterations:   61
  Time/Iter:    1.8868 s/iter

Training FAST Method (Max Matrix Size: 500 x 500)...
  Total Time:   94.43 seconds
  Iterations:   50
  Time/Iter:    1.8886 s/iter

--- PERFORMANCE WINNER ---
Slow Method is 1.0x faster PER ITERATION.
Fast Method finished overall training 1.2x faster.
